In [1]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
from pathlib import Path

TARGET_COLUMN = 'OT'
T_IN = 96  
T_OUT = 24 

FILE_PATH = Path('Data') / 'ETTh1.csv' 

if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"ERROR: Dataset not found! Please ensure 'ETTh1.csv' is located in the '{FILE_PATH.parent}/' directory."
    )
else:
    print(f"File found and confirmed at: {FILE_PATH}")

print(f"Lookback (T_in): {T_IN}, Forecast Horizon (T_out): {T_OUT}")

File found and confirmed at: Data\ETTh1.csv
Lookback (T_in): 96, Forecast Horizon (T_out): 24


In [2]:
df = pd.read_csv(FILE_PATH, parse_dates=['date'])
df.set_index('date', inplace=True)

print("Initial DataFrame Head:")
print(df.head())

print("\nMissing values check:")
print(df.isnull().sum())


Initial DataFrame Head:
                      HUFL   HULL   MUFL   MULL   LUFL   LULL         OT
date                                                                    
2016-07-01 00:00:00  5.827  2.009  1.599  0.462  4.203  1.340  30.531000
2016-07-01 01:00:00  5.693  2.076  1.492  0.426  4.142  1.371  27.787001
2016-07-01 02:00:00  5.157  1.741  1.279  0.355  3.777  1.218  27.787001
2016-07-01 03:00:00  5.090  1.942  1.279  0.391  3.807  1.279  25.044001
2016-07-01 04:00:00  5.358  1.942  1.492  0.462  3.868  1.279  21.948000

Missing values check:
HUFL    0
HULL    0
MUFL    0
MULL    0
LUFL    0
LULL    0
OT      0
dtype: int64


In [3]:
def check_stationarity(timeseries):
    print('Results of ADF Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','Lags Used','No. of Observations'])
    for key, value in dftest[4].items():
        dfoutput['Critical Value (%s)' % key] = value
    print(dfoutput)
    
    if dfoutput['p-value'] > 0.05:
        print("\nConclusion: Time Series is NON-STATIONARY (p > 0.05). Differencing is required.")
        return False
    else:
        print("\nConclusion: Time Series is STATIONARY (p <= 0.05). No differencing needed.")
        return True

check_stationarity(df[TARGET_COLUMN].dropna())


if not check_stationarity(df[TARGET_COLUMN].dropna()):
    print("\nApplying first-order differencing to the target variable...")
    df['OT_diff'] = df[TARGET_COLUMN].diff()


Results of ADF Test:
Test Statistic             -3.487964
p-value                     0.008302
Lags Used                  44.000000
No. of Observations     17375.000000
Critical Value (1%)        -3.430726
Critical Value (5%)        -2.861706
Critical Value (10%)       -2.566859
dtype: float64

Conclusion: Time Series is STATIONARY (p <= 0.05). No differencing needed.
Results of ADF Test:
Test Statistic             -3.487964
p-value                     0.008302
Lags Used                  44.000000
No. of Observations     17375.000000
Critical Value (1%)        -3.430726
Critical Value (5%)        -2.861706
Critical Value (10%)       -2.566859
dtype: float64

Conclusion: Time Series is STATIONARY (p <= 0.05). No differencing needed.


In [4]:
if 'OT_diff' in df.columns:
    df.dropna(inplace=True) 

df.dropna(inplace=True) 

scaler = MinMaxScaler(feature_range=(0, 1))

features_to_scale = df.columns
df_scaled_np = scaler.fit_transform(df[features_to_scale])
df_scaled = pd.DataFrame(df_scaled_np, columns=features_to_scale, index=df.index)

target_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler.fit(df[[TARGET_COLUMN]])

print("\nScaled DataFrame Head:")
print(df_scaled.head())

F = df_scaled.shape[1]
print(f"\nNumber of Features (F): {F}")


Scaled DataFrame Head:
                         HUFL      HULL      MUFL      MULL      LUFL  \
date                                                                    
2016-07-01 00:00:00  0.615599  0.454943  0.628980  0.467510  0.556576   
2016-07-01 01:00:00  0.612708  0.459449  0.626458  0.464878  0.550279   
2016-07-01 02:00:00  0.601143  0.436920  0.621438  0.459689  0.512595   
2016-07-01 03:00:00  0.599698  0.450437  0.621438  0.462320  0.515693   
2016-07-01 04:00:00  0.605480  0.450437  0.626458  0.467510  0.521990   

                         LULL        OT  
date                                     
2016-07-01 00:00:00  0.613765  0.691018  
2016-07-01 01:00:00  0.620783  0.636233  
2016-07-01 02:00:00  0.586144  0.636233  
2016-07-01 03:00:00  0.599955  0.581468  
2016-07-01 04:00:00  0.599955  0.519656  

Number of Features (F): 7


In [5]:
target_feature_index = df_scaled.columns.get_loc(TARGET_COLUMN)

def create_sequences(data_df, T_in, T_out, target_idx):
    """
    Creates Encoder Input (X) and Decoder Target (Y) sequences.
    
    X: (N, T_in, F) - All features for the lookback window.
    Y: (N, T_out, 1) - Only the target feature for the forecast horizon.
    """
    X, Y = [], []
    data_np = data_df.values 

    for i in range(len(data_np) - T_in - T_out + 1):
        
        X_seq = data_np[i : i + T_in, :]
        X.append(X_seq)
        
        Y_seq = data_np[i + T_in : i + T_in + T_out, target_idx]
        Y.append(Y_seq)

    return np.array(X), np.array(Y).reshape(-1, T_out, 1)

X_all, Y_all = create_sequences(df_scaled, T_IN, T_OUT, target_feature_index)

print(f"\nFinal Shape of Encoder Input (X_all): {X_all.shape}")
print(f"Final Shape of Decoder Target (Y_all): {Y_all.shape}")


Final Shape of Encoder Input (X_all): (17301, 96, 7)
Final Shape of Decoder Target (Y_all): (17301, 24, 1)


In [6]:
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

N = X_all.shape[0] # Total number of samples

train_end = int(train_ratio * N)
val_end = int((train_ratio + val_ratio) * N)

X_train, Y_train = X_all[:train_end], Y_all[:train_end]
X_val, Y_val = X_all[train_end:val_end], Y_all[train_end:val_end]
X_test, Y_test = X_all[val_end:], Y_all[val_end:]

def create_decoder_input(N_samples, T_out, F):
    return np.zeros((N_samples, T_out, F))

decoder_input_train = create_decoder_input(X_train.shape[0], T_OUT, F)
decoder_input_val = create_decoder_input(X_val.shape[0], T_OUT, F)
decoder_input_test = create_decoder_input(X_test.shape[0], T_OUT, F)

print("\n--- Final Dataset Shapes ---")
print(f"X_train (Encoder Input): {X_train.shape}")
print(f"Y_train (Decoder Target): {Y_train.shape}")
print(f"Decoder_Input_train: {decoder_input_train.shape}")
print("-" * 30)
print(f"X_test (Test Encoder Input): {X_test.shape}")
print(f"Y_test (Test Decoder Target): {Y_test.shape}")


--- Final Dataset Shapes ---
X_train (Encoder Input): (12110, 96, 7)
Y_train (Decoder Target): (12110, 24, 1)
Decoder_Input_train: (12110, 24, 7)
------------------------------
X_test (Test Encoder Input): (2596, 96, 7)
Y_test (Test Decoder Target): (2596, 24, 1)


In [7]:
import os
import numpy as np
import pickle

RESULTS_FOLDER = 'Results' 

os.makedirs(RESULTS_FOLDER, exist_ok=True)

print("Saving final NumPy arrays...")
np.save(f'{RESULTS_FOLDER}/X_train.npy', X_train)
np.save(f'{RESULTS_FOLDER}/Y_train.npy', Y_train)
np.save(f'{RESULTS_FOLDER}/X_val.npy', X_val)
np.save(f'{RESULTS_FOLDER}/Y_val.npy', Y_val)
np.save(f'{RESULTS_FOLDER}/decoder_input_train.npy', decoder_input_train)
np.save(f'{RESULTS_FOLDER}/decoder_input_val.npy', decoder_input_val)
np.save(f'{RESULTS_FOLDER}/X_test.npy', X_test)
np.save(f'{RESULTS_FOLDER}/Y_test.npy', Y_test)
np.save(f'{RESULTS_FOLDER}/decoder_input_test.npy', decoder_input_test)

with open(f'{RESULTS_FOLDER}/target_scaler.pkl', 'wb') as file:
    pickle.dump(target_scaler, file)

print(f"All data arrays and scaler successfully saved to {RESULTS_FOLDER}/ folder.")

Saving final NumPy arrays...
All data arrays and scaler successfully saved to Results/ folder.


In [8]:
print("--- NaN Count Check ---")
print(df.isnull().sum())
print("-" * 30)

--- NaN Count Check ---
HUFL    0
HULL    0
MUFL    0
MULL    0
LUFL    0
LULL    0
OT      0
dtype: int64
------------------------------
